In [7]:
!python run_v2_pipeline.py \
  --mrn-file /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_v2/DFCI_MRN_to_check.csv \
  --output-dir /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_v2 \
  --max-workers 8 \
  --bundle-max-tokens 6000 \
  # --overwrite-existing

Existing output-dir state: mode=resume labels_mrns=120 checkpoint_mrns=121 failed_mrns=1
Running: /data/gusev/USERS/jpconnor/conda/envs/DFCI_GPT_LLM/bin/python /data/gusev/USERS/jpconnor/code/CAIA/COMPASS/PROFILE/v2/prepare_event_candidates.py --output-dir /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_v2 --mrn-file /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_v2/DFCI_MRN_to_check.csv
Selecting v2 notes: 100%|█████████████████████| 135/135 [00:02<00:00, 61.38it/s]
Wrote candidate notes: /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_v2/LLM_v2_candidate_text_data.csv
Wrote patient context: /data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_v2/LLM_v2_patient_context.csv
Patients in context: 135
Patients with selected notes: 135
Selected notes: 5827
Text source: compiled
Requested MRNs: 569
Running: /data/gusev/USERS/jpconnor/conda/envs/DFCI_GPT_LLM/bin/python /data/gusev/USERS/jpconnor/code/CAIA/COMPASS/PROFILE/v2/generate_event_labels.py --output-dir /data/gusev/USERS/jpconnor/dat

In [3]:
import pandas as pd 
import os 

DATA_PATH = '/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_v2/'

original_list = pd.read_csv(os.path.join(DATA_PATH, 'original_MRN_list.csv'))
original_list.columns = ['DFCI_MRN']

new_labels = pd.read_csv(os.path.join(DATA_PATH, 'LLM_v2_generated_labels.tsv'), sep='\t')
new_labels = new_labels.loc[new_labels['DFCI_MRN'].isin(original_list['DFCI_MRN'])]

new_labels['neuroendocrine_history'] = new_labels['ever_histologies'].apply(lambda x : ('neuroendocrine' in str(x)) or ('small_cell' in str(x)))

new_labels = new_labels[['DFCI_MRN', 'metastatic_date', 'metastatic_sites', 'current_histology',
                         'ever_histologies', 'first_platinum_date', 'first_platinum_source', 'neuroendocrine_history',
                         'transformation_suspected_date', 'transformation_confirmed_date', 'adt_nonresponse_present',
                         'adt_nonresponse_reasons', 'other_cancer_present', 'biomarker_flags', 'trial_context_mentioned',
                         'confidence', 'supporting_quotes', 'supporting_quote_dates']].sort_values(by='neuroendocrine_history', ascending=False)

new_labels.to_csv(os.path.join(DATA_PATH, 'LLM_histology_reports.tsv'), index=False, sep='\t')